# 05 复习：张量、广播、数据预处理

这一节用简短回答和 Python 例子复习三个问题：

- 张量是什么？
- 广播是什么？
- 数据为什么要预处理？

## 1. 张量是什么？

张量可以理解为一个可以存放数字的多维数组，是深度学习中表示数据的基本结构。

- 0 维张量：一个数字，也叫标量，例如 `3.14`
- 1 维张量：一排数字，也叫向量，例如 `[1, 2, 3]`
- 2 维张量：表格或矩阵，例如 3 行 4 列的数据
- 更高维张量：可以表示图片、语音、文本批次等复杂数据

在 PyTorch 中，模型的输入、参数、输出通常都是张量。

In [ ]:
import torch

x = torch.arange(12).reshape(3, 4)

print(x)
print("形状:", x.shape)
print("维度数:", x.ndim)
print("元素个数:", x.numel())

## 2. 广播是什么？

广播是一种自动扩展张量形状的规则。两个形状不同的张量做运算时，只要它们的维度兼容，PyTorch 会自动把较小的张量扩展成合适的形状，再进行逐元素计算。

广播的好处是：不用手动复制数据，也能写出简洁的向量化代码。

常见规则：从最后一个维度开始比较，如果两个维度相等，或其中一个维度是 `1`，就可以广播。

In [ ]:
a = torch.arange(3).reshape(3, 1)
b = torch.arange(2).reshape(1, 2)

print("a 的形状:", a.shape)
print(a)
print("b 的形状:", b.shape)
print(b)

result = a + b
print("a + b 的形状:", result.shape)
print(result)

## 3. 数据为什么要预处理？

真实数据通常不能直接喂给模型，因为它可能存在缺失值、文本类别、量纲差异、异常值等问题。数据预处理的目的，是把原始数据整理成模型能够理解、能够稳定学习的张量或表格。

常见预处理包括：

- 处理缺失值：删除、填充均值、填充默认类别等
- 处理类别特征：把文本类别转换成数字，例如独热编码
- 数值缩放：让不同特征的数值范围更接近
- 拆分数据：把特征 `X` 和标签 `y` 分开

预处理做得好，模型更容易训练，结果也更可靠。

In [1]:
import pandas as pd
from io import StringIO

raw_data = """NumRooms,Alley,Price
NA,Pave,127500
2,NA,106000
4,NA,178100
NA,NA,140000
"""

data = pd.read_csv(StringIO(raw_data), na_values="NA")
print("原始数据:")
print(data)

inputs = data.drop(columns="Price")
outputs = data["Price"]

# 类别列转成数字列，缺失类别单独作为一列。
inputs = pd.get_dummies(inputs, dummy_na=True)

# 数值列用平均值填充缺失值。
inputs = inputs.fillna(inputs.mean(numeric_only=True))

print("\n预处理后的输入:")
print(inputs)
print("\n标签:")
print(outputs)

原始数据:
   NumRooms Alley   Price
0       NaN  Pave  127500
1       2.0   NaN  106000
2       4.0   NaN  178100
3       NaN   NaN  140000

预处理后的输入:
   NumRooms  Alley_Pave  Alley_nan
0       3.0        True      False
1       2.0       False       True
2       4.0       False       True
3       3.0       False       True

标签:
0    127500
1    106000
2    178100
3    140000
Name: Price, dtype: int64


## 小结

- 张量：深度学习中的多维数组，用来表示数据、参数和计算结果。
- 广播：让形状兼容的张量自动扩展后进行逐元素运算。
- 数据预处理：把真实世界中不规则的数据整理成模型能学习的格式。